Community Assistanty to allow Backed by a Firebase Database

Business Justification: I have a community management solution built by me (https://audiatur.co). In simple terms, Audiatur provides all the tools(organising elections, conference management, event calendar, general info storage) needed to manage a community(a community could be a church, association etc).

One key requirement is to allow community members get info about the community using Whatsapp. This project makes that possible. Data is stored on Firebase

First the imports

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import firebase_admin
from firebase_admin import credentials, firestore
from google.cloud.firestore_v1.base_query import FieldFilter
import json
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
import gradio as gr
from datetime import datetime
from tools import check_if_user_is_registered_for_event, get_user_registered_events, register_user_for_event

Lets setup model

In [ ]:

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

now lets setup Firebase so we can load data

In [ ]:
config = json.loads(os.getenv("FIREBASE_CONFIG_JSON"))
cred = credentials.Certificate(config)
try:
    firebase_admin.initialize_app(cred)
    print("Firebase initialized successfully!")
except ValueError:
    print("App already initialized.")



Load communities and conferences

In [ ]:
db = firestore.client()
event_registrations_collection = db.collection('event_registration')
projectsCollection = db.collection('projects')
query = projectsCollection.where(filter=FieldFilter("projectType", "in", ["COMMUNITY", "CONFERENCE"]))
projects = query.get()
project_map = {p.id: p for p in projects}


In [ ]:
def get_project_hierarchy(project):
    chain = []
    parent_id = project.to_dict().get("parentProjectId")
    if not parent_id:
        return []
    current =  project_map.get(parent_id).to_dict()
    while current:
        chain.append(current.get("name"))
        parent_id = current.get("parentProjectId")

        if not parent_id:
            break

        current = project_map.get(parent_id).to_dict()

    return list(reversed(chain))

In [ ]:
def create_content(data):
    if data.get('projectType') == 'CONFERENCE':
        return f"""
        Event: {data.get('name')}
        eventId: {data.get('id')}
        Short Description: {data.get('shortDescription')}
        Description: {data.get('description')}
        Venue: {data.get('venue')}
        Start Date: { data.get("startDate").isoformat() if data.get("startDate") else "" }
        End Date: { data.get("endDate").isoformat() if data.get("endDate") else "" }
        Event Type: { data.get('projectType') }
        Status: { data.get('status') }
        organizer hierarchy: {" > ".join(get_project_hierarchy(data))}   
        """
    else :
        return f"""
        Community: {data.get('name')}
        Description: {data.get('description')}
        Short Description: {data.get('shortDescription')}
        Community hierarchy: {" > ".join(get_project_hierarchy(data))}   
        """


Lets create Langchain documents

In [ ]:
documents = [Document(page_content=create_content(project), metadata={'source': project.id}) for project in projects]
for d in documents:
    print(d.page_content)

Lets chunk the data

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

Lets create our vector and vector db

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print(embeddings)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


Lets setup tools

In [ ]:



def handle_tool_call(tool_call):
    tool_name = tool_call.get("name")
    args = tool_call.get("args") or {}

    try:
        if tool_name == "check_if_user_is_registered_for_event":
            return check_if_user_is_registered_for_event(
                event_registrations_collection, 
                project_map,
                args.get("user_email"), 
                args.get("event_id")
            )
        elif tool_name == "get_user_registered_events":
            return get_user_registered_events(
                event_registrations_collection, 
                project_map,
                args.get("user_email")
            )
        elif tool_name == "register_user_for_event":
            event_registration_result = register_user_for_event(
                event_registrations_collection, 
                project_map,
                args.get("user_email"), 
                args.get("event_id")
            )
            if event_registration_result.get("success"):
                print("event registered successfully: call the mailer here and pass only what is needed, put mailer code in mailer.py")
            return event_registrations_collection
        else:
            return f"Unknown or unsupported tool: {tool_name}"
    except Exception as e:
        return f"Tool error ({tool_name}): {e}"


Lets test out tools

In [ ]:
print(check_if_user_is_registered_for_event(event_registrations_collection,project_map,'adeyemiadetayo07@gmail.com',"XHeqdyHZoMgUECp0Iuhm"))   
print(get_user_registered_events(event_registrations_collection,project_map,'adeyemiadetayo07@gmail.com'))
print(register_user_for_event(event_registrations_collection,project_map,'adeyemiadetayo07@gmail.com',"XHeqdyHZoMgUECp0Iuhm"))

Lets start retrieving

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)
llm = llm.bind_tools([check_if_user_is_registered_for_event, get_user_registered_events, register_user_for_event])  
today = datetime.utcnow().strftime("%A, %B %d, %Y")
SYSTEM_PROMPT_TEMPLATE = f"""
You are a knowledgeable, friendly assistant that helps users get information about a community.

You can:
- answer questions about events
- check if a user is registered for an event
- register users for events
- retrieve events a user registered for

Today's date is: {today} (UTC).

Use this date when answering questions about months, dates, upcoming events, or past events.

IMPORTANT TOOL RULES:

1. Tools require specific parameters.
2. NEVER call a tool if required parameters are missing.
3. If the user's email is missing, ask the user for their email first.
4. If an event ID is missing, find the event ID from the context before calling the tool.
5. Users may mention an event name instead of the event ID. If so, find the corresponding event ID from the context.
6. Only call tools when you have ALL required parameters.

If relevant, use the given context to answer any question.
If you don't know the answer, say so.

Context:
{{context}}
"""


Lets setup chat function

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage,ToolMessage,ToolCall


def answer_question(question: str, history):
    docs = retriever.invoke(question)
    print(docs)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    for user_msg, bot_msg in history:
        if user_msg:
            messages.append(HumanMessage(content=user_msg))
        if bot_msg:
            messages.append(AIMessage(content=bot_msg))

    # add the latest user message
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    print(response.content)
    while response.tool_calls:
        messages.append(response)
        for tool_call in response.tool_calls:
            tool_call_result = handle_tool_call(tool_call)
            print(tool_call_result)
            messages.append(
                ToolMessage(
                    content=str(tool_call_result),
                    tool_call_id=tool_call["id"],
                    name=tool_call["name"],   
                )
            )
        response = llm.invoke(messages)

    return response.content

Lets wire up Gradio

In [ ]:
gr.ChatInterface(answer_question).launch()